# Лабораторная работа №3. Задание 5
## Приложение Panel App: интерактивная панель «Чаевые в ресторане»

Ноутбук оформлен как приложение **Panel**: ячейки с компонентами, помеченные `.servable()`, составляют панель, которая запускается командой

```bash
panel serve panel_app.ipynb --show
```

Для данных используется набор `tips` из seaborn (чек, чаевые, пол, курение, день недели, время суток, размер компании).

## Используемые утилиты, библиотеки и зависимости

In [ ]:
import sys, panel, bokeh, pandas, seaborn, matplotlib, numpy
print('Python', sys.version.split()[0])
for m in (panel, bokeh, pandas, seaborn, matplotlib, numpy): print(m.__name__, m.__version__)

## 1. Данные

In [ ]:
import numpy as np, pandas as pd, seaborn as sns
import matplotlib
matplotlib.use('agg')
from matplotlib.figure import Figure
import panel as pn
pn.extension('tabulator', sizing_mode='stretch_width')

tips = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv')
tips['tip_pct'] = (tips['tip'] / tips['total_bill'] * 100).round(2)   # чаевые в % от чека
tips.head()

## 2. Виджеты
Фильтры по дню недели, времени суток, курильщикам, размеру компании и диапазону суммы чека, а также выбор признаков для диаграммы рассеяния.

In [ ]:
days    = pn.widgets.CheckButtonGroup(name='День недели', options=['Thur','Fri','Sat','Sun'], value=['Thur','Fri','Sat','Sun'], button_type='primary')
times   = pn.widgets.CheckBoxGroup(name='Время суток', options=['Lunch','Dinner'], value=['Lunch','Dinner'], inline=True)
smoker  = pn.widgets.RadioButtonGroup(name='Курильщики', options=['Все','Yes','No'], value='Все')
size    = pn.widgets.IntRangeSlider(name='Размер компании', start=1, end=6, value=(1,6), step=1)
bill    = pn.widgets.RangeSlider(name='Сумма чека, $', start=float(tips.total_bill.min()), end=float(tips.total_bill.max()),
                                 value=(float(tips.total_bill.min()), float(tips.total_bill.max())), step=1)
color_by = pn.widgets.Select(name='Цвет точек', options=['day','time','sex','smoker'], value='day')
y_axis   = pn.widgets.Select(name='Ось Y', options=['tip','tip_pct'], value='tip')

## 3. Логика: фильтрация и построение графиков
Функции связываются с виджетами через `pn.bind` — при изменении любого виджета пересчитываются только зависящие от него компоненты.

In [ ]:
def filtered(days, times, smoker, size, bill):
    df = tips[tips.day.isin(days) & tips.time.isin(times)]
    if smoker != 'Все':
        df = df[df.smoker == smoker]
    df = df[df['size'].between(*size) & df.total_bill.between(*bill)]
    return df

def scatter(days, times, smoker, size, bill, color_by, y_axis):
    df = filtered(days, times, smoker, size, bill)
    fig = Figure(figsize=(7, 4.5)); ax = fig.subplots()
    for key, g in df.groupby(color_by, observed=True):
        ax.scatter(g.total_bill, g[y_axis], label=str(key), alpha=.8, s=35)
    ax.set_xlabel('Сумма чека, $'); ax.set_ylabel('Чаевые, $' if y_axis=='tip' else 'Чаевые, % от чека')
    ax.set_title(f'Чаевые и сумма чека (строк: {len(df)})'); ax.grid(alpha=.3)
    if len(df): ax.legend(title=color_by)
    return pn.pane.Matplotlib(fig, tight=True, dpi=110)

def by_day(days, times, smoker, size, bill):
    df = filtered(days, times, smoker, size, bill)
    fig = Figure(figsize=(7, 4.5)); ax = fig.subplots()
    order = [d for d in ['Thur','Fri','Sat','Sun'] if d in set(df.day)]
    m = df.groupby('day', observed=True)['tip'].mean().reindex(order)
    ax.bar(m.index.astype(str), m.values, color='#4c78a8')
    ax.set_xlabel('День недели'); ax.set_ylabel('Средние чаевые, $'); ax.set_title('Средние чаевые по дням'); ax.grid(axis='y', alpha=.3)
    return pn.pane.Matplotlib(fig, tight=True, dpi=110)

def stats(days, times, smoker, size, bill):
    df = filtered(days, times, smoker, size, bill)
    if df.empty:
        return pn.pane.Alert('Нет данных для выбранных фильтров', alert_type='warning')
    return pn.Row(pn.indicators.Number(name='Заказов', value=len(df), format='{value}'),
                  pn.indicators.Number(name='Средний чек, $', value=round(df.total_bill.mean(),2), format='{value}'),
                  pn.indicators.Number(name='Средние чаевые, %', value=round(df.tip_pct.mean(),2), format='{value}'))

def table(days, times, smoker, size, bill):
    return pn.widgets.Tabulator(filtered(days, times, smoker, size, bill), pagination='local', page_size=10, show_index=False, disabled=True)

args = dict(days=days, times=times, smoker=smoker, size=size, bill=bill)

## 4. Компоновка: боковая панель и вкладки
`pn.template.FastListTemplate` задаёт заголовок и боковую панель с фильтрами; основная область — вкладки «Обзор», «Диаграмма» и «Данные».

In [ ]:
tabs = pn.Tabs(
    ('Обзор',    pn.Column(pn.bind(stats, **args), pn.bind(by_day, **args))),
    ('Диаграмма', pn.Column(pn.Row(color_by, y_axis), pn.bind(scatter, **args, color_by=color_by, y_axis=y_axis))),
    ('Данные',   pn.bind(table, **args)),
    dynamic=True)

app = pn.template.FastListTemplate(
    title='Чаевые в ресторане — Panel App',
    sidebar=[pn.pane.Markdown('### Фильтры'), days, times, smoker, size, bill],
    main=[tabs], accent='#4c78a8')
app.servable();

## 5. Запуск, совместный доступ и публикация
* Локально: `panel serve panel_app.ipynb --show` (откроется http://localhost:5006/panel_app).
* **Share** в Jupyter/Anaconda Notebooks: *Share → Publish/Share link* позволяет выдать доступ другим пользователям к ноутбуку и опубликованной панели; в данной среде (без доступа к облачному сервису) эта функция недоступна, поэтому совместный доступ обеспечивается публикацией репозитория и статической сборкой панели (см. `github_pages_steps.md`).
* Статическая версия для GitHub Pages: `panel convert panel_app.ipynb --to pyodide-worker --out docs`.